# Marketstack — Financial Products & Corporate Data

Demonstrates **commodities**, **bonds**, **ETFs**, **dividends**, **splits**,
**company ratings**, and **SEC EDGAR** tools
using the [Marketstack API](https://marketstack.com).

Requires a `MARKETSTACK_ACCESS_KEY` environment variable set in `.env`.

---
## Setup

In [ ]:
!pip install -q -e "../.."
!pip install -q -r ../requirements.txt

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from demos.shared.llm_setup import get_llm
from pymcpx.marketstack import (
    MarketstackBondsTool,
    MarketstackCommoditiesTool,
    MarketstackCompanyRatingsTool,
    MarketstackDividendsTool,
    MarketstackEdgarCikTool,
    MarketstackEdgarFactsTool,
    MarketstackEdgarSubmissionsTool,
    MarketstackEtfsTool,
    MarketstackSplitsTool,
)

llm = get_llm()

tools = [
    MarketstackCommoditiesTool(),
    MarketstackBondsTool(),
    MarketstackEtfsTool(),
    MarketstackDividendsTool(),
    MarketstackSplitsTool(),
    MarketstackCompanyRatingsTool(),
    MarketstackEdgarCikTool(),
    MarketstackEdgarSubmissionsTool(),
    MarketstackEdgarFactsTool(),
]

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a financial data assistant. Use the marketstack tools to answer "
            "questions about financial products and company data.\n\n"
            "Available tools:\n"
            "- marketstack_commodities(commodity_name, date_from, date_to, frequency) \u2014 real-time or historical commodity prices (e.g. gold, crude_oil)\n"
            "- marketstack_bonds(country, limit, offset) \u2014 government bond yields for leading countries\n"
            "- marketstack_etfs(ticker, list_type, date_from, date_to, limit, offset) \u2014 list ETFs or get holdings for a specific ETF\n"
            "- marketstack_dividends(symbols, symbol, date_from, date_to, sort, limit, offset) \u2014 dividend payment history\n"
            "- marketstack_splits(symbols, symbol, date_from, date_to, sort, limit, offset) \u2014 stock split history\n"
            "- marketstack_company_ratings(ticker, date_from, date_to, rated) \u2014 analyst buy/sell/hold ratings and price targets\n"
            "- marketstack_edgar_cik(company_name, cik_code, limit, offset) \u2014 search CIK codes by company name or look up by CIK\n"
            "- marketstack_edgar_submissions(cik_code, cik_code_name, report_from, report_to, filing_from, filing_to, accession_number) \u2014 SEC EDGAR filing history\n"
            "- marketstack_edgar_facts(cik_code, query_type, frame, unit) \u2014 company facts from SEC EDGAR",
        ),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
print("\u2705 Agent ready. Running scenarios...")

In [ ]:
result = executor.invoke({"input": "Get the current commodity price for gold and crude oil."})
print(result["output"])

In [ ]:
result = executor.invoke(
    {
        "input": "Show me analyst ratings for AAPL and check for any recent dividends or stock splits."
    }
)
print(result["output"])

In [ ]:
result = executor.invoke({"input": "Find the CIK code for Microsoft Corporation."})
print(result["output"])

In [ ]:
result = executor.invoke(
    {
        "input": "What are the current government bond yields for the United States? Also list the first 5 available ETFs."
    }
)
print(result["output"])

In [ ]:
result = executor.invoke(
    {
        "input": "Look up the EDGAR facts for Apple (CIK: 0000320193) and tell me what data is available."
    }
)
print(result["output"])